In [0]:
count_before = spark.table("md_bronze.sales_orders").count()

from pyspark.sql.functions import input_file_name

bronze_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.schemaLocation", "/tmp/sales_input/_schema")
         .option("header", "true")
         .load("/FileStore/sales_input")
         .withColumn(
             "source_file",
             input_file_name()
         )
)

(
    bronze_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            "/tmp/checkpoints/sales_bronze"
        )
        .trigger(availableNow=True)
        .toTable("md_bronze.sales_orders")
)

query = (
    bronze_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            "/tmp/checkpoints/sales_bronze"
        )
        .trigger(availableNow=True)
        .toTable("md_bronze.sales_orders")
)

query.awaitTermination()
print(query.lastProgress)
count_after = spark.table("md_bronze.sales_orders").count()

print(f"Before load: {count_before}")
print(f"After load:  {count_after}")
print(f"New rows:    {count_after - count_before}")


In [0]:
dbutils.fs.rm(
    "/tmp/checkpoints/sales_bronze",
    True
)
dbutils.fs.rm(
    "/tmp/sales_input/_schema",
    True
)

spark.sql("""
DROP TABLE IF EXISTS md_bronze.sales_orders
""")

print("✅ sales_orders dropped")



display(
    dbutils.fs.ls("/tmp/checkpoints/sales_bronze")
)

display(
    dbutils.fs.ls("/tmp/sales_input/")
)

In [0]:
%sql
SELECT
    source_file,
    COUNT(*)
FROM md_bronze.sales_orders
GROUP BY source_file
ORDER BY source_file;

In [0]:
%sql SHOW DATABASES;
USE md_bronze;

SHOW TABLES;

In [0]:
import json

print(
    json.dumps(
        query.lastProgress,
        indent=2
    )
)

In [0]:
display(dbutils.fs.ls("/FileStore/sales_input"))